In [ ]:
import cosmic
from cosmic.output import load_initC
import h5py as h5
import os
import pandas as pd
import astropy.units as u
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from importlib import reload
from astropy.coordinates import SkyCoord


import legwork as lw
import gala.potential as gp
import cogsworth

import sys
sys.path.append("../helpers")

import detections
import const

plt.rc('font', family='serif')
plt.rcParams['text.usetex'] = False
fs = 24

# update various fontsizes to match
params = {'figure.figsize': (12, 8),
          'legend.fontsize': 0.7*fs,
          'legend.title_fontsize': 0.8*fs,
          'axes.labelsize': fs,
          'xtick.labelsize': 0.9 * fs,
          'ytick.labelsize': 0.9 * fs,
          'axes.linewidth': 1.1,
          'xtick.major.size': 7,
          'xtick.minor.size': 4,
          'ytick.major.size': 7,
          'ytick.minor.size': 4}
plt.rcParams.update(params)

%config InlineBackend.figure_format = 'retina'

# Load data

In [ ]:
dco_types = const.DCO_TYPES.copy()
for i in range(len(dco_types)):
    dco_types.append(f"{dco_types[i]}_pessimistic")

In [ ]:
pops = {
    dco_type: cogsworth.pop.load(f"/mnt/ceph/users/twagg/lisa-dcos/fiducial/{dco_type}_in_band.h5") for dco_type in dco_types
}

In [ ]:
for dco_type in dco_types:
    pop = pops[dco_type]
    print(dco_type)
    pop.final_pos, pop.final_vel

BHBH
BHNS
BHWD
NSNS


NSWD
BHBH_pessimistic
BHNS_pessimistic
BHWD_pessimistic
NSNS_pessimistic
NSWD_pessimistic


In [ ]:
detectable_pops = {}
for dco_type in dco_types:
    mask = (
        (pops[dco_type].bpp["snr_lisa_10yr_final_pos"] > 7)
        | (pops[dco_type].bpp["snr_lisa_10yr_initial_pos"] > 7)
        | (pops[dco_type].bpp["snr_decigo_10yr_final_pos"] > 7)
        | (pops[dco_type].bpp["snr_decigo_10yr_initial_pos"] > 7)
    )
    detectable_pops[dco_type] = pops[dco_type][mask]

cogsworth warning: You've just masked a population that wasn't fully loaded from a file. This means that the masked population won't have access to the parts that were not loaded. If you don't need the missing parts then this is fine, for reference those are: ['orbits'].
cogsworth warning: You've just masked a population that wasn't fully loaded from a file. This means that the masked population won't have access to the parts that were not loaded. If you don't need the missing parts then this is fine, for reference those are: ['orbits'].
cogsworth warning: You've just masked a population that wasn't fully loaded from a file. This means that the masked population won't have access to the parts that were not loaded. If you don't need the missing parts then this is fine, for reference those are: ['orbits'].
cogsworth warning: You've just masked a population that wasn't fully loaded from a file. This means that the masked population won't have access to the parts that were not loaded. If y

In [ ]:
f_detect_dfs = {
    dco_type: pd.read_hdf(f"/mnt/ceph/users/twagg/lisa-dcos/fiducial/{dco_type}_f_detect.h5", key="f_detect") for dco_type in dco_types
}

In [ ]:
detectable_sources = {
    dco_type: detectable_pops[dco_type].to_legwork_sources(assume_mw_galactocentric=True) for dco_type in dco_types
}

In [ ]:
for dco_type in dco_types:
    print(dco_type)
    detectable_sources[dco_type].n_proc = 28
    detectable_sources[dco_type].evolve_sources(t_evol=detectable_pops[dco_type].initial_galaxy.tau - detectable_pops[dco_type].bpp["tphys"].values * u.Myr)

BHBH


BHNS
BHWD
NSNS
NSWD
BHBH_pessimistic
BHNS_pessimistic
BHWD_pessimistic
NSNS_pessimistic
NSWD_pessimistic


In [ ]:
for dco_type in dco_types:

    p = detectable_pops[dco_type]

    initial_distance = SkyCoord(
        x=p.initial_galaxy.x, y=p.initial_galaxy.y, z=p.initial_galaxy.z,
        v_x=p.initial_galaxy.v_x, v_y=p.initial_galaxy.v_y, v_z=p.initial_galaxy.v_z,
        representation_type="cartesian", unit=u.kpc, frame="galactocentric"
    ).icrs.distance

    instruments = ["LISA", "DECIGO"]
    positions = [("initial_pos", initial_distance),
                 ("final_pos", p.get_final_mw_skycoord().icrs.distance)]

    dur = 4 * u.yr
    dur_name = "4yr"

    for instrument in instruments:
        for pos_name, pos in positions:
            print(dco_type, instrument, pos_name, dur_name)
            label = f"{instrument.lower()}_{dur_name}_{pos_name}"
            detectable_sources[dco_type].sc_params["instrument"] = instrument
            detectable_sources[dco_type].sc_params["t_obs"] = dur
            detectable_sources[dco_type].dist = pos
            snr = detectable_sources[dco_type].get_snr()
            detectable_pops[dco_type].bpp[f"snr_{label}"] = snr
            pops[dco_type].bpp[f"snr_{label}"] = np.zeros(len(pops[dco_type].bpp), dtype=float)
            pops[dco_type].bpp.loc[detectable_pops[dco_type].bpp.index, f"snr_{label}"] = snr

            base_compare = "lisa_10yr_final_pos"
            detection_ratio = (
                detectable_pops[dco_type].bpp[detectable_pops[dco_type].bpp[f"snr_{label}"] > 7].groupby("MW_instance", observed=False)["weights"].sum()
                / detectable_pops[dco_type].bpp[detectable_pops[dco_type].bpp[f"snr_{base_compare}"] > 7].groupby("MW_instance")["weights"].sum()
            )

            f_detect_dfs[dco_type][f"{label}"] = f_detect_dfs[dco_type][f"{base_compare}"] * detection_ratio

    f_detect_dfs[dco_type].fillna(0, inplace=True)

BHBH LISA initial_pos 4yr
BHBH LISA final_pos 4yr
BHBH DECIGO initial_pos 4yr
BHBH DECIGO final_pos 4yr
BHNS LISA initial_pos 4yr
BHNS LISA final_pos 4yr
BHNS DECIGO initial_pos 4yr
BHNS DECIGO final_pos 4yr
BHWD LISA initial_pos 4yr
BHWD LISA final_pos 4yr
BHWD DECIGO initial_pos 4yr
BHWD DECIGO final_pos 4yr
NSNS LISA initial_pos 4yr
NSNS LISA final_pos 4yr
NSNS DECIGO initial_pos 4yr
NSNS DECIGO final_pos 4yr
NSWD LISA initial_pos 4yr
NSWD LISA final_pos 4yr
NSWD DECIGO initial_pos 4yr
NSWD DECIGO final_pos 4yr
BHBH_pessimistic LISA initial_pos 4yr
BHBH_pessimistic LISA final_pos 4yr
BHBH_pessimistic DECIGO initial_pos 4yr
BHBH_pessimistic DECIGO final_pos 4yr
BHNS_pessimistic LISA initial_pos 4yr
BHNS_pessimistic LISA final_pos 4yr
BHNS_pessimistic DECIGO initial_pos 4yr
BHNS_pessimistic DECIGO final_pos 4yr
BHWD_pessimistic LISA initial_pos 4yr
BHWD_pessimistic LISA final_pos 4yr
BHWD_pessimistic DECIGO initial_pos 4yr
BHWD_pessimistic DECIGO final_pos 4yr
NSNS_pessimistic LISA in

In [ ]:
for dco_type in dco_types:
    # pops[dco_type].bpp.to_hdf(f"/mnt/ceph/users/twagg/lisa-dcos/fiducial/{dco_type}_in_band.h5", key="bpp")
    f_detect_dfs[dco_type].to_hdf(f"/mnt/ceph/users/twagg/lisa-dcos/fiducial/{dco_type}_f_detect.h5", key="f_detect", mode="w")